<a href="https://colab.research.google.com/github/Faustino790/ColabProject/blob/main/Project_Arcane_1_0V.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


```
███╗   ███╗██╗███╗   ██╗███████╗ ██████╗ ██████╗ ██╗      █████╗ ██████╗
████╗ ████║██║████╗  ██║██╔════╝██╔════╝██╔═══██╗██║     ██╔══██╗██╔══██╗
██╔████╔██║██║██╔██╗ ██║█████╗  ██║     ██║   ██║██║     ███████║██████╔╝
██║╚██╔╝██║██║██║╚██╗██║██╔══╝  ██║     ██║   ██║██║     ██╔══██║██╔══██╗
██║ ╚═╝ ██║██║██║ ╚████║███████╗╚██████╗╚██████╔╝███████╗██║  ██║██████╔╝
╚═╝     ╚═╝╚═╝╚═╝  ╚═══╝╚══════╝ ╚═════╝ ╚═════╝ ╚══════╝╚═╝  ╚═╝╚═════╝
```

#  **SERVER STARTER**

In [ ]:
import os
import re
import json
from google.colab import drive

# Step 1: Create update.sh script
def CRD():
    script_content = """#!/bin/bash
b='\\033[1m'
r='\\E[31m'
g='\\E[32m'
c='\\E[36m'
endc='\\E[0m'
enda='\\033[0m'

printf "\\n\\n$c$b    Software Updating... $endc$enda" >&2
if sudo apt-get update &> /dev/null
then
    printf "\\r$g$b    Latest Software Installed.. $endc$enda\\n" >&2
else
    printf "\\r$r$b    Error Occurred $endc$enda\\n" >&2
    exit
fi
"""
    with open('update.sh', 'w') as script:
        script.write(script_content)

    os.system("chmod +x update.sh")
    os.system("./update.sh")

CRD()

# Step 2: Mount Google Drive
drive.mount('/content/drive')

# __Step 3: Change Directory__
os.chdir("/content/drive/My Drive/minecraft-server")
!ls

# Step 4: Install neofetch to show system info
!sudo apt install neofetch -y &> /dev/null
!neofetch

# Step 5: Load or create colabconfig
if os.path.isfile("colabconfig.json"):
  with open("colabconfig.json") as f:
    try:
        colabconfig = json.load(f)
    except json.JSONDecodeError:
        colabconfig = {}  # If file is broken

# Set default keys if missing
if "server_type" not in colabconfig:
    colabconfig["server_type"] = "generic"
if "server_version" not in colabconfig:
    colabconfig["server_version"] = "1.20.6"

# Save back updated config
with open("colabconfig.json", "w") as f:
    json.dump(colabconfig, f)

# Step 6: Install Java (with proper version parsing)
server_version = colabconfig["server_version"]
version_tuple = tuple(map(int, server_version.split(".")))

if colabconfig["server_type"] == "forge" and version_tuple < (1, 17):
    !sudo apt-get install openjdk-15-jre-headless -y &> /dev/null && echo "OpenJDK 15 installed."
else:
    !sudo apt-get install openjdk-21-jre-headless -y &> /dev/null && echo "OpenJDK 21 installed."

# Step 7: Java version check
import subprocess
java_version_output = subprocess.run(["java", "-version"], stderr=subprocess.PIPE, text=True)
print(java_version_output.stderr)

if "21" in java_version_output.stderr:
    print("✅ OpenJDK 21 is working fine.")
else:
    print("⚠ Java 21 not detected. Minecraft 1.17+ may not run properly.")

#===============#7.5: Install/Verify Node.js ========================

print("\n🔧 Installing Node.js (if missing)...")

node_check = subprocess.run(["which", "node"], stdout=subprocess.PIPE, text=True)
if node_check.stdout.strip() == "":
    print("⏳ Node.js not found. Installing Node.js...")
    !curl -fsSL https://deb.nodesource.com/setup_18.x | sudo -E bash - &> /dev/null
    !sudo apt-get install -y nodejs &> /dev/null
else:
    print("✅ Node.js already installed.")

node_version = subprocess.run(["node", "--version"], stdout=subprocess.PIPE, text=True)
npm_version = subprocess.run(["npm", "--version"], stdout=subprocess.PIPE, text=True)
print("🟢 Node.js version:", node_version.stdout.strip())
print("🟠 NPM version:", npm_version.stdout.strip())

#=====================================================================

#============================= Denger Zone ============================

jar_list = {
    'paper': 'server.jar',
    'fabric': 'fabric-server-launch.jar',
    'generic': 'server.jar',
    'forge': 'forge.jar'
}
jar_name = jar_list[colabconfig["server_type"]]


if colabconfig["server_type"] == "paper":
    server_flags = "-XX:+UseG1GC -XX:+ParallelRefProcEnabled -XX:MaxGCPauseMillis=200 " \
                   "-XX:+UnlockExperimentalVMOptions -XX:+DisableExplicitGC -XX:+AlwaysPreTouch " \
                   "-XX:G1NewSizePercent=30 -XX:G1MaxNewSizePercent=40 -XX:G1HeapRegionSize=8M " \
                   "-XX:G1ReservePercent=20 -XX:G1HeapWastePercent=5 -XX:G1MixedGCCountTarget=4 " \
                   "-XX:InitiatingHeapOccupancyPercent=15 -XX:G1MixedGCLiveThresholdPercent=90 " \
                   "-XX:G1RSetUpdatingPauseTimePercent=5 -XX:SurvivorRatio=32 -XX:+PerfDisableSharedMem " \
                   "-XX:MaxTenuringThreshold=1 -Dusing.aikars.flags=https://mcflags.emc.gs " \
                   "-Daikars.new.flags=true"
else:
    server_flags = ""

#=======================================================================

# ========== #10: Dynamic RAM Allocation ==========

meminfo = subprocess.run(["grep", "MemTotal", "/proc/meminfo"], stdout=subprocess.PIPE, text=True)
mem_kb = int(re.search(r'\d+', meminfo.stdout).group())
mem_gb = mem_kb // 1024 // 1024
xmx = int(mem_gb * 0.85)
xms = 2 if xmx > 4 else 1
memory_allocation = f"-Xms{xms}G -Xmx{xmx}G"
print(f"🚀 Auto RAM Allocation: {memory_allocation}")

# ================== #12: Run Server ===============

!java {memory_allocation} {server_flags} -jar {jar_name} nogui

# ================ #11: Tunnel selection ==============

tunnel = colabconfig.get("tunnel_service", "none").lower()

if tunnel == "playit":
    print("🌐 Using Playit for tunneling...")

elif tunnel == "ngrok":
    print("🌐 Using Ngrok for tunneling...")

else:
    print("⚠️ No tunneling active or unknown service:", tunnel)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
banned-ips.json      FancyAnalytics   server.jar	    wepif.yml
banned-players.json  help.yml	      server.properties     whitelist.json
bukkit.yml	     libraries	      spigot.yml	    world
cache		     logs	      tunnel		    world_nether
colabconfig.json     ops.json	      update.sh		    world_the_end
commands.yml	     permissions.yml  usercache.json
config		     playit	      version_history.json
eula.txt	     plugins	      versions
            .-/+oossssoo+/-.
        `:+ssssssssssssssssss+:`
      -+ssssssssssssssssssyyssss+-
    .ossssssssssssssssssdMMMNysssso.
   /ssssssssssshdmmNNmmyNMMMMhssssss/
  +ssssssssshmydMMMMMMMNddddyssssssss+
 /sssssssshNMMMyhhyyyyhmNMMMNhssssssss/
.ssssssssdMMMNhsssssssssshNMMMdssssssss.
+sssshhhyNMMNyssssssssssssyNMMMysssssss+
ossyNMMMNyMMhsssssssssssssshmmmhssssssso
ossyNMMMNyMMhsssssssssssssshmmmhssssssso
+sssshhhyNMMNyssssss

# BASIC CONSOLE

In [ ]:
# Enable command blocks in server.properties
with open("server.properties", "r") as f:
    lines = f.readlines()

with open("server.properties", "w") as f:
    for line in lines:
        if line.startswith("enable-command-block"):
            f.write("enable-command-block=true\n")
        else:
            f.write(line)

print("✅ Command blocks enabled!")

✅ Command blocks enabled!


In [ ]:
import json

ops_data = [
    {"uuid" : "abc-123", "name": "Luffy",
"level": 4},
    {"uuid" : "def-456", "name": "Zoro",
"level": 4},
    {"uuid" : "ghi-789", "name": "Sanji",
"level": 3},
]
with open("ops.json",  "r") as f:
  json.dump(ops_data, f)

UnsupportedOperation: not writable

In [ ]:
# Show list of current OPs from ops.json
import json

with open("ops.json", "r") as f:
    ops = json.load(f)

print("✅ Current Operators:")
for op in ops:
    print("- " + op["name"])

✅ Current Operators:
- Jackcraft_Gaming
- D99T
- FI4E_GAMER_yt


In [ ]:
import json

# 👤 Yahan player ka naam daal jise OP se hatana hai
player_to_remove = "...."  # <-- change this

# Load ops.json
try:
    with open("ops.json", "r") as f:
        ops = json.load(f)

    # Filter out the player
    new_ops = [op for op in ops if op["name"].lower() != player_to_remove.lower()]

    # Save updated list
    with open("ops.json", "w") as f:
        json.dump(new_ops, f, indent=4)

    if len(ops) == len(new_ops):
        print(f"⚠️ {player_to_remove} was not in the OP list.")
    else:
        print(f"✅ {player_to_remove} removed from OP list.")
except FileNotFoundError:
    print("❌ ops.json file not found.")

✅ Expand_Gaming removed from OP list.


In [ ]:
import json
import uuid
import os

# 👤 Yahan player ka naam daal jise OP banana hai
player_to_add = "FaustinoGamer709"  # <-- change this

# Default OP level (4 is full admin)
op_level = 4

# Load existing ops.json
ops_path = "ops.json"
if os.path.exists(ops_path):
    with open(ops_path, "r") as f:
        ops = json.load(f)
else:
    ops = []

# Check if player already in OP list
if any(op["name"].lower() == player_to_add.lower() for op in ops):
    print(f"⚠️ {player_to_add} is already an operator.")
else:
    # Generate UUID (offline mode safe)
    player_uuid = str(uuid.uuid3(uuid.NAMESPACE_DNS, player_to_add)).replace("-", "")

    new_op = {
        "uuid": player_uuid,
        "name": player_to_add,
        "level": op_level,
        "bypassesPlayerLimit": False
    }

    ops.append(new_op)

    # Save updated list
    with open(ops_path, "w") as f:
        json.dump(ops, f, indent=4)

    print(f"✅ {player_to_add} has been added as an operator.")

✅ FaustinoGamer709 has been added as an operator.


# **🖥️SERVER CONSOLE🖥️**

In [ ]:
#@title Package Installer { vertical-output: true }
run = False #@param {type:"boolean"}
#@markdown *Package management actions (gasp)*
action = "Check Installed" #@param ["Install", "Check Installed", "Remove"] {allow-input: true}

package = "wget" #@param {type:"string"}
system = "apt" #@param ["apt", ""]

def install(package=package, system=system):
  if system == "apt":
    !apt --fix-broken install > /dev/null 2>&1
    !killall apt > /dev/null 2>&1
    !rm /var/lib/dpkg/lock-frontend
    !dpkg --configure -a > /dev/null 2>&1

    !apt-get  install -o Dpkg::Options::="--force-confold" --no-install-recommends -y $package

    !dpkg --configure -a > /dev/null 2>&1
    !apt  update > /dev/null 2>&1

    !apt install $package > /dev/null 2>&1

def check_installed(package=package, system=system):
  if system == "apt":
    !apt list --installed | grep $package

def remove(package=package, system=system):
  if system == "apt":
    !apt remove $package

if run:
  if action == "Install":
    install()
  if action == "Check Installed":
    check_installed()
  if action == "Remove":
    remove()

In [ ]:
#@title **Advance_Connectivity 📊**
!curl -sSf https://sshx.io/get | sh
!sshx